# IS 4487 Assignment 9: Customer Segmentation with Clustering

In this assignment, you will:
- Apply unsupervised learning to explore patterns in hotel booking behavior
- Use K-Means and Gaussian Mixture Models (GMM) for customer segmentation
- Evaluate model quality with metrics like Silhouette Score and Davies-Bouldin Index
- Connect clustering to actionable business insights

## Why This Matters

Businesses like hotels and travel platforms (e.g., Airbnb or Expedia) rely on customer segmentation to tailor promotions, pricing strategies, and service levels. Unlike supervised models, clustering helps uncover patterns when no labels exist—an ideal tool when entering new markets or analyzing unstructured customer behavior.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_09_clustering.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.

## 1. Setup and Load Data

### Business framing:  

### Do the following:
Before we can cluster or segment anything, we need clean, accessible data in a usable format.

- Import the necessary Python libraries
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Display the first few rows

### In Your Response:
1. What stands out in the initial preview? Any columns or rows that seem unusual?

In [2]:
# Add code here 🔧
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('hotels.csv')

df.head()

/tmp/ipykernel_238/541189808.py:6: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('hotels.csv')


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0.0,Transient,0.0,0.0,0.0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0.0,Transient,0.0,0.0,0.0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0.0,Transient,75.0,0.0,0.0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0.0,Transient,75.0,0.0,0.0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0.0,Transient,98.0,0.0,1.0,Check-Out,2015-07-03


### ✍️ Your Response: 🔧
1. The agent row seems to have many null values, but there are also many cloumns that contain a significant

## 2. Select and Prepare Features

### Business framing:  

A hotel might want to group guests based on how long they stay, how far in advance they book, or how likely they are to make special requests. You need to pick variables that represent meaningful guest behavior.

### Do the following:
- Choose 3–5 numeric features related to customer behavior
- Drop missing values if needed
- Standardize using `StandardScaler`

### In Your Response:
1. What features did you select and why?
2. What kinds of patterns or segments do you expect to find?


In [8]:
# Add code here 🔧
from sklearn.preprocessing import StandardScaler

features = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
            'previous_cancellations', 'total_of_special_requests']

X = df[features]

X = X.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

import pandas as pd
X_scaled = pd.DataFrame(X_scaled, columns=features)

X_scaled.head()

NameError: name 'df' is not defined

### ✍️ Your Response: 🔧
1. I choose lead time, stays in weekend nights, stays in week nights, previous cancellations, and total of special requests as these variables allowed for the most detail from clustering.

2. I expect to see some differences in the way the the clusters are formed in the based on the format of the data.

## 3. Apply K-Means Clustering

### Business framing:  

Let’s say you’re working with the hotel’s marketing manager. She wants to group guests into a few clear types to target email campaigns. K-Means is a fast, simple way to try this.

### Do the following:
- Fit a `KMeans` model with your selected features
- Choose a value of `k` (e.g. 3, 4, or 5)
- Predict clusters and assign to each guest
- Visualize using a scatterplot of 2 features

Much of this assignment has already been covered in the lab. Please be sure to complete the lab before the assignment.

### In Your Response:
1. What `k` value did you choose, and how did you decide?
2. What types of customers seem to show up in the clusters?



In [4]:
# Add code here 🔧
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

k = 4
kmeans = KMeans(n_clusters=k, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df_cleaned = df.dropna(subset=['lead_time', 'stays_in_weekend_nights',
                               'stays_in_week_nights', 'previous_cancellations',
                               'total_of_special_requests'])
df_cleaned['cluster'] = clusters

plt.figure(figsize=(8,6))
for i in range(k):
    cluster_data = df_cleaned[df_cleaned['cluster'] == i]
    plt.scatter(cluster_data['lead_time'], cluster_data['total_of_special_requests'], label=f'Cluster {i}')
plt.xlabel('Lead Time (days)')
plt.ylabel('Total Special Requests')
plt.title('K-Means Clustering of Hotel Guests')
plt.legend()
plt.show()

NameError: name 'X_scaled' is not defined

### ✍️ Your Response: 🔧
1. I choose k value 4 as it seemed to be the best fin this use case.

2. The clustering seems to be missing some component that does not allow me to see what the data visualization should look like.


## 4. Apply Gaussian Mixture Model (GMM)

### Business framing:  

Not all guests fit neatly into one cluster. GMM lets us capture uncertainty — useful if customers behave similarly across groups.

### Do the following:
- Fit a GMM with the same number of clusters you chose in Part 3
- Predict soft clusters (remember that soft clustering deals with probabilities, not labels)
- Visualize the GMM model so that you may compare it to the KMeans scatterplot

### In Your Response:
1. How did the GMM results compare to KMeans?
2. What business questions might GMM help answer better?


In [5]:
# Add your code here
from sklearn.mixture import GaussianMixture
import numpy as np
import matplotlib.pyplot as plt

gmm = GaussianMixture(n_components=k, random_state=42)
gmm.fit(X_scaled)

probabilities = gmm.predict_proba(X_scaled)

df_cleaned['gmm_cluster'] = np.argmax(probabilities, axis=1)

plt.figure(figsize=(8,6))
for i in range(k):
    cluster_data = df_cleaned[df_cleaned['gmm_cluster'] == i]
    plt.scatter(cluster_data['lead_time'], cluster_data['total_of_special_requests'], label=f'GMM Cluster {i}', alpha=0.7)
plt.xlabel('Lead Time (days)')
plt.ylabel('Total Special Requests')
plt.title('GMM Clustering of Hotel Guests')
plt.legend()
plt.show()

NameError: name 'X_scaled' is not defined

### ✍️ Your Response: 🔧
1. GMM clustered the data in a different formating but at th eend of the day the visualization resulted in the same outcome.

2. It would better explain the cancelations compare to Kmeans.


## 5. Evaluate Your Models

### Business framing:  

In business, models should be both useful and reliable. You’ll compare model quality using standard evaluation metrics.

### Do the following:
- Calculate the following **for each** of the models:
  - WCSS
  - Silhouette Score
  - Davies-Bouldin Index

  **NOTE:** This step may take up to 5 minutes.  It is a lot of computation time.  Please be patient.  Or you can limit the scores to using a random sample of 10K rows.

**Remember**:
- Lower WCSS = tighter, better-defined clusters
- Silhouette score ranges from -1 to 1.  Higher values = better clustering
- Lower Davies-Boulding Index = better clustering

### In Your Response:
1. Which model performed better on the metrics?
2. Would you recommend KMeans or GMM for a business analyst? Why?


In [7]:
# Add code here 🔧
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import numpy as np

sample_size = 10000
if len(X_scaled) > sample_size:
    X_eval = X_scaled.sample(sample_size, random_state=42)
    km_labels_eval = df_cleaned.loc[X_eval.index, 'cluster']
    gmm_labels_eval = df_cleaned.loc[X_eval.index, 'gmm_cluster']
else:
    X_eval = X_scaled
    km_labels_eval = df_cleaned['cluster']
    gmm_labels_eval = df_cleaned['gmm_cluster']
wcss = KMeans(n_clusters=k, random_state=42).fit(X_eval).inertia_
sil_score = silhouette_score(X_eval, km_labels_eval)
db_index = davies_bouldin_score(X_eval, km_labels_eval)

print("K-Means Evaluation Metrics:")
print(f"WCSS: {wcss:.2f}")
print(f"Silhouette Score: {sil_score:.3f}")
print(f"Davies-Bouldin Index: {db_index:.3f}")

gmm = GaussianMixture(n_components=k, random_state=42)
gmm.fit(X_eval)

gmm_labels = gmm.predict(X_eval)

gmm_sil_score = silhouette_score(X_eval, gmm_labels)

gmm_db_index = davies_bouldin_score(X_eval, gmm_labels)

gmm_centers = gmm.means_
wcss_gmm = 0
for i, center in enumerate(gmm_centers):
    wcss_gmm += np.sum((X_eval.values[gmm_labels == i] - center)**2)

print("\nGMM Evaluation Metrics:")
print(f"WCSS: {wcss_gmm:.2f}")
print(f"Silhouette Score: {gmm_sil_score:.3f}")
print(f"Davies-Bouldin Index: {gmm_db_index:.3f}")

NameError: name 'X_scaled' is not defined

### ✍️ Your Response: 🔧
1. There is not enought data for me to declare that one model preformed objectively better in this circumstance.

2. I would recommend Kmenas for a business analysis as it is a simpler representation of the dataset.


## 6. Business Interpretation

### Business framing:  

What do these clusters mean in the real world? Could they represent solo travelers, families, or bargain shoppers?

### Do the following:
- Display the characteristics of each cluster (e.g. average `lead_time`, `special_requests`)
- Sort the clusters to make the differences more clear

### In Your Response:
1. What do the segments represent in terms of guest behavior?
2. How could the hotel tailor services or promotions to each group?


In [6]:
# Add code here 🔧
behavior_features = ['lead_time', 'stays_in_weekend_nights',
                     'stays_in_week_nights', 'previous_cancellations',
                     'total_of_special_requests']

km_profile = df_cleaned.groupby('cluster')[behavior_features].mean().sort_values('lead_time')
print("K-Means Cluster Profiles:")
display(km_profile)

gmm_profile = df_cleaned.groupby('gmm_cluster')[behavior_features].mean().sort_values('lead_time')
print("GMM Cluster Profiles:")
display(gmm_profile)

NameError: name 'df_cleaned' is not defined

### ✍️ Your Response: 🔧
1. The segments in this case represent a customer segment that roughly behaves in a way that can be group as one action.

2. This could help tailor the experience to each customer group.


## 7. Final Reflection

### Business framing:  

Many teams ask for "segmentation" without knowing how it works. You now have hands-on experience with two clustering techniques and how to present the results.

### In Your Response:
1. What was most challenging about unsupervised learning?
2. When would you use clustering instead of supervised models?
3. How would you explain the value of clustering to a non-technical manager?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. It is difficult to understand the outputs as we are not working with one specific goal in mind when observing the data.

2. The clustering lets one see the similarities in customer behavior and lets you adapt to those actions.

3. The value of clustering is in seeing the patterns that emerge and not in the raw data. The visualizations are the goal when using unsupervised learning.

4. It is realted to ability to now understand more modeling techiques that can be put into practical applications.

## Submission Instructions

✅ **Before submitting:**
- Make sure all code cells are run and outputs are visible  
- All markdown questions are answered thoughtfully  
- Submit the assignment as an **HTML file** on Canvas


In [9]:
!jupyter nbconvert --to html "assignment_09_AldenEverett.ipynb"

[NbConvertApp] Converting notebook assignment_09_AldenEverett.ipynb to html
[NbConvertApp] Writing 339080 bytes to assignment_09_AldenEverett.html
